# 04 - Gold Layer Transformation

## Purpose

This notebook transforms Silver layer census tables into
business-ready dimensional models using a Star Schema design.

The Gold layer supports:

- BI dashboards
- Workforce analysis
- Occupational trends
- Industrial trends
- District comparisons

The warehouse consists of:

Dimensions:
- dim_geography
- dim_occupation
- dim_community
- dim_worker_type
- dim_industry

Facts:
- fact_occupation_workforce
- fact_industrial_workforce

### 1 — Imports

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

### 2 — Load Silver Tables

In [0]:
df_st = spark.read.table(
    "pyspark_dbt.silver.st_main_workers"
)

df_sc = spark.read.table(
    "pyspark_dbt.silver.sc_main_workers"
)

df_ind_main = spark.read.table(
    "pyspark_dbt.silver.industrial_main_workers"
)

df_ind_marg = spark.read.table(
    "pyspark_dbt.silver.industrial_marginal_workers"
)

print("Silver tables loaded successfully.")

# Dimension 1 — Geography


In [0]:
dim_geography = (

    df_st.select(
        "State_Code",
        "District_Code",
        col("State_District").alias("District_Name"),
        "Geo_Level"
    )

    .distinct()
)

### 4 — Surrogate Key

In [0]:
window_spec = Window.orderBy(
    "State_Code",
    "District_Code"
)

dim_geography = (

    dim_geography

    .withColumn(
        "Geo_Key",
        row_number().over(window_spec)
    )

    .select(
        "Geo_Key",
        "State_Code",
        "District_Code",
        "District_Name",
        "Geo_Level"
    )

)

### 5 — Save Geography Dimension

In [0]:
(
    dim_geography.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "pyspark_dbt.gold.dim_geography"
    )
)

print("dim_geography created.")

# Dimension 2 — Occupation

In [0]:
occupation_sc_st = (

    df_st.select(
        "Division",
        "Sub_Division",
        "Group",
        "Family",
        "NCO_Name",
        "Hierarchy_Level"
    )

    .union(

        df_sc.select(
            "Division",
            "Sub_Division",
            "Group",
            "Family",
            "NCO_Name",
            "Hierarchy_Level"
        )
    )

)

In [0]:
occupation_industrial = (

    df_ind_main.select(
        "Division",
        "Sub_Division",
        lit(None).cast("string").alias("Group"),
        lit(None).cast("string").alias("Family"),
        "NCO_Name",
        "Hierarchy_Level"
    )

    .union(

        df_ind_marg.select(
            "Division",
            "Sub_Division",
            lit(None).cast("string").alias("Group"),
            lit(None).cast("string").alias("Family"),
            "NCO_Name",
            "Hierarchy_Level"
        )
    )

)

In [0]:
dim_occupation = (

    occupation_sc_st

    .union(
        occupation_industrial
    )

    .distinct()
)

In [0]:
window_spec = Window.orderBy(
    "Division",
    "Sub_Division",
    "Group",
    "Family"
)

dim_occupation = (

    dim_occupation

    .withColumn(
        "Occupation_Key",
        row_number().over(window_spec)
    )

    .select(
        "Occupation_Key",
        "Division",
        "Sub_Division",
        "Group",
        "Family",
        "NCO_Name",
        "Hierarchy_Level"
    )

)

In [0]:
window_spec = Window.orderBy(
    "Division",
    "Sub_Division",
    "Group",
    "Family"
)

dim_occupation = (

    dim_occupation

    .withColumn(
        "Occupation_Key",
        row_number().over(window_spec)
    )

    .select(
        "Occupation_Key",
        "Division",
        "Sub_Division",
        "Group",
        "Family",
        "NCO_Name",
        "Hierarchy_Level"
    )

)

In [0]:
(
    dim_occupation.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "pyspark_dbt.gold.dim_occupation"
    )
)

print("dim_occupation created.")

# Dimension 3 — Community

In [0]:
dim_community = (

    df_st.select(
        "Community_Category"
    )

    .union(

        df_sc.select(
            "Community_Category"
        )

    )

    .distinct()
)

In [0]:
window_spec = Window.orderBy(
    "Community_Category"
)

dim_community = (

    dim_community

    .withColumn(
        "Community_Key",
        row_number().over(window_spec)
    )

    .select(
        "Community_Key",
        "Community_Category"
    )
)

In [0]:
(
    dim_community.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "pyspark_dbt.gold.dim_community"
    )
)

print("dim_community created.")

# Dimension 4 — Worker Type

In [0]:
dim_worker_type = (

    df_ind_main.select(
        "Worker_Type"
    )

    .union(
        df_ind_marg.select(
            "Worker_Type"
        )
    )

    .distinct()
)

In [0]:
window_spec = Window.orderBy(
    "Worker_Type"
)

dim_worker_type = (

    dim_worker_type

    .withColumn(
        "Worker_Type_Key",
        row_number().over(window_spec)
    )

    .select(
        "Worker_Type_Key",
        "Worker_Type"
    )
)

In [0]:
(
    dim_worker_type.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "pyspark_dbt.gold.dim_worker_type"
    )
)

print("dim_worker_type created.")

# Dimension 5 — Industry

In [0]:
industry_data = [

    ("A","Agriculture, Forestry and Fishing"),
    ("B","Mining and Quarrying"),
    ("C_HHI","Manufacturing (HHI)"),
    ("C_NON_HHI","Manufacturing (Non HHI)"),
    ("D_E","Utilities"),
    ("F","Construction"),
    ("G_HHI","Trade (HHI)"),
    ("G_NON_HHI","Trade (Non HHI)"),
    ("H","Transportation and Storage"),
    ("I","Accommodation and Food Services"),
    ("J_HHI","Information and Communication (HHI)"),
    ("J_NON_HHI","Information and Communication (Non HHI)"),
    ("K_M","Finance, Real Estate and Professional Services"),
    ("N_O","Administration and Public Services"),
    ("P_Q","Education and Healthcare"),
    ("R_U_HHI","Other Services (HHI)"),
    ("R_U_NON_HHI","Other Services (Non HHI)")
]

In [0]:
dim_industry = spark.createDataFrame(
    industry_data,
    [
        "Industry_Code",
        "Industry_Name"
    ]
)

In [0]:
window_spec = Window.orderBy(
    "Industry_Code"
)

dim_industry = (

    dim_industry

    .withColumn(
        "Industry_Key",
        row_number().over(window_spec)
    )

    .select(
        "Industry_Key",
        "Industry_Code",
        "Industry_Name"
    )
)

In [0]:
(
    dim_industry.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "pyspark_dbt.gold.dim_industry"
    )
)

print("dim_industry created.")

## Fact Table 1 — Occupation Workforce

Business Process:

How many workers exist for a given:

- Geography
- Occupation
- Community

Measures:

- Total Persons
- Total Males
- Total Females
- Rural Persons
- Rural Males
- Rural Females
- Urban Persons
- Urban Males
- Urban Females

### 21 — Combine ST and SC datasets

In [0]:
occupation_fact_base = (

    df_st

    .unionByName(
        df_sc
    )

)

### 22 — Join Geography Dimension

In [0]:
occupation_fact = (

    occupation_fact_base.alias("fact")

    .join(

        dim_geography.alias("geo"),

        (
            col("fact.State_Code") == col("geo.State_Code")
        )
        &
        (
            col("fact.District_Code") == col("geo.District_Code")
        ),

        "left"

    )

)

### 23 — Join Occupation Dimension

In [0]:
occupation_fact = (

    occupation_fact.alias("fact")

    .join(

        dim_occupation.alias("occ"),

        (
            col("fact.Division") == col("occ.Division")
        )
        &
        (
            col("fact.Sub_Division") == col("occ.Sub_Division")
        )
        &
        (
            col("fact.Group") == col("occ.Group")
        )
        &
        (
            col("fact.Family") == col("occ.Family")
        )
        &
        (
            col("fact.NCO_Name") == col("occ.NCO_Name")
        ),

        "left"

    )

)

### 24 — Join Community Dimension

In [0]:
occupation_fact = (

    occupation_fact.alias("fact")

    .join(

        dim_community.alias("comm"),

        col("fact.Community_Category")
        ==
        col("comm.Community_Category"),

        "left"

    )

)

### 25 — Select Final Fact Columns

In [0]:
fact_occupation_workforce = (

    occupation_fact.select(

        col("Geo_Key"),

        col("Occupation_Key"),

        col("Community_Key"),

        col("Total___Persons")
            .alias("Total_Persons"),

        col("Total___Males")
            .alias("Total_Males"),

        col("Total___Females")
            .alias("Total_Females"),

        col("Rural___Persons")
            .alias("Rural_Persons"),

        col("Rural___Males")
            .alias("Rural_Males"),

        col("Rural___Females")
            .alias("Rural_Females"),

        col("Urban___Persons")
            .alias("Urban_Persons"),

        col("Urban___Males")
            .alias("Urban_Males"),

        col("Urban___Females")
            .alias("Urban_Females")

    )

)

### 26 — Fact Validation

In [0]:
print("=" * 70)

print("FACT OCCUPATION VALIDATION")

print("=" * 70)

print(
    f"Rows : {fact_occupation_workforce.count()}"
)

print()

print(
    f"Null Geography Keys : "
    f"{fact_occupation_workforce.filter(col('Geo_Key').isNull()).count()}"
)

print(
    f"Null Occupation Keys : "
    f"{fact_occupation_workforce.filter(col('Occupation_Key').isNull()).count()}"
)

print(
    f"Null Community Keys : "
    f"{fact_occupation_workforce.filter(col('Community_Key').isNull()).count()}"
)

### Save Fact Table

In [0]:
(
    fact_occupation_workforce.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "pyspark_dbt.gold.fact_occupation_workforce"
    )
)

print(
    "fact_occupation_workforce created."
)

### 28 — Verify

In [0]:
display(
    fact_occupation_workforce
)

In [0]:
fact_occupation_workforce.count()

## Fact Table 2 — Industrial Workforce

Business Process:

How many workers exist for a given:

- Geography
- Occupation
- Worker Type
- Industry

Measures:

- Persons
- Males
- Females

### 29 — Combine Industrial Main and Marginal

In [0]:
industrial_base = (

    df_ind_main

    .unionByName(
        df_ind_marg
    )

)

### 30 — Create Industry Mapping Dictionary

In [0]:
industry_mapping = {

    "A": (
        "Industrial_Category___A__Excluding_Cultivators__Agricultural_Labourers____Persons",
        "Industrial_Category___A__Excluding_Cultivators__Agricultural_Labourers____Males",
        "Industrial_Category___A__Excluding_Cultivators__Agricultural_Labourers____Females"
    ),

    "B": (
        "Industrial_Category___B___Persons",
        "Industrial_Category___B___Males",
        "Industrial_Category___B___Females"
    ),

    "C_HHI": (
        "Industrial_Category___C___HHI___Persons",
        "Industrial_Category___C___HHI___Males",
        "Industrial_Category___C___HHI___Females"
    ),

    "C_NON_HHI": (
        "Industrial_Category___C___Non_HHI___Persons",
        "Industrial_Category___C___Non_HHI___Males",
        "Industrial_Category___C___Non_HHI___Females"
    ),

    "D_E": (
        "Industrial_Category___D___E___Persons",
        "Industrial_Category___D___E___Males",
        "Industrial_Category___D___E___Females"
    ),

    "F": (
        "Industrial_Category___F___Persons",
        "Industrial_Category___F___Males",
        "Industrial_Category___F___Females"
    ),

    "G_HHI": (
        "Industrial_Category___G___HHI___Persons",
        "Industrial_Category___G___HHI___Males",
        "Industrial_Category___G___HHI___Females"
    ),

    "G_NON_HHI": (
        "Industrial_Category___G___Non_HHI___Persons",
        "Industrial_Category___G___Non_HHI___Males",
        "Industrial_Category___G___Non_HHI___Females"
    ),

    "H": (
        "Industrial_Category___H___Persons",
        "Industrial_Category___H___Males",
        "Industrial_Category___H___Females"
    ),

    "I": (
        "Industrial_Category___I___Persons",
        "Industrial_Category___I___Males",
        "Industrial_Category___I___Females"
    ),

    "J_HHI": (
        "Industrial_Category___J___HHI___Persons",
        "Industrial_Category___J___HHI___Males",
        "Industrial_Category___J___HHI___Females"
    ),

    "J_NON_HHI": (
        "Industrial_Category___J___Non_HHI___Persons",
        "Industrial_Category___J___Non_HHI___Males",
        "Industrial_Category___J___Non_HHI___Females"
    ),

    "K_M": (
        "Industrial_Category___K_to_M___Persons",
        "Industrial_Category___K_to_M___Males",
        "Industrial_Category___K_to_M___Females"
    ),

    "N_O": (
        "Industrial_Category___N_to_O___Persons",
        "Industrial_Category___N_to_O___Males",
        "Industrial_Category___N_to_O___Females"
    ),

    "P_Q": (
        "Industrial_Category___P_to_Q___Persons",
        "Industrial_Category___P_to_Q___Males",
        "Industrial_Category___P_to_Q___Females"
    ),

    "R_U_HHI": (
        "Industrial_Category___R_to_U___HHI___Persons",
        "Industrial_Category___R_to_U___HHI___Males",
        "Industrial_Category___R_to_U___HHI___Females"
    ),

    "R_U_NON_HHI": (
        "Industrial_Category___R_to_U___Non_HHI___Persons",
        "Industrial_Category___R_to_U___Non_HHI___Males",
        "Industrial_Category___R_to_U___Non_HHI___Females"
    )
}

### 31 — Unpivot Industries

In [0]:
industry_frames = []

for industry_code, cols_tuple in industry_mapping.items():

    persons_col, males_col, females_col = cols_tuple

    temp_df = (

        industrial_base

        .select(
            "State_Code",
            "District_Code",
            "Division",
            "Sub_Division",
            "NCO_Name",
            "Worker_Type",

            lit(industry_code).alias("Industry_Code"),

            col(persons_col).alias("Persons"),
            col(males_col).alias("Males"),
            col(females_col).alias("Females")
        )

    )

    industry_frames.append(temp_df)

### 32 — Combine All Industries

In [0]:
from functools import reduce

industrial_long = reduce(
    lambda x, y: x.unionByName(y),
    industry_frames
)

### 33 — Join Geography

In [0]:
industrial_fact = (

    industrial_long.alias("fact")

    .join(
        dim_geography.alias("geo"),

        (
            col("fact.State_Code")
            ==
            col("geo.State_Code")
        )
        &
        (
            col("fact.District_Code")
            ==
            col("geo.District_Code")
        ),

        "left"
    )

)

### 34 — Join Occupation

In [0]:
industrial_fact = (

    industrial_fact.alias("fact")

    .join(

        dim_industrial_occupation.alias("occ"),

        (
            col("fact.Division")
            ==
            col("occ.Division")
        )
        &
        (
            col("fact.Sub_Division")
            ==
            col("occ.Sub_Division")
        )
        &
        (
            col("fact.NCO_Name")
            ==
            col("occ.NCO_Name")
        ),

        "left"

    )

)

### 35 — Join Worker Type

In [0]:
industrial_fact = (

    industrial_fact.alias("fact")

    .join(

        dim_worker_type.alias("wt"),

        col("fact.Worker_Type")
        ==
        col("wt.Worker_Type"),

        "left"

    )

)

### 36 — Join Industry Dimension

In [0]:
industrial_fact = (

    industrial_fact.alias("fact")

    .join(

        dim_industry.alias("ind"),

        col("fact.Industry_Code")
        ==
        col("ind.Industry_Code"),

        "left"

    )

)

### 37 — Final Fact Selection

In [0]:
fact_industrial_workforce = (

    industrial_fact.select(

        "Geo_Key",
        "Industrial_Occupation_Key",
        "Worker_Type_Key",
        "Industry_Key",

        "Persons",
        "Males",
        "Females"

    )

)

### 38 — Validation

In [0]:
print("="*70)
print("FACT INDUSTRIAL VALIDATION")
print("="*70)

print(
    f"Rows: {fact_industrial_workforce.count()}"
)

print(
    f"Null Geo Keys: "
    f"{fact_industrial_workforce.filter(col('Geo_Key').isNull()).count()}"
)

print(
    f"Null Occupation Keys: "
    f"{fact_industrial_workforce.filter(col('Industrial_Occupation_Key').isNull()).count()}"
)

print(
    f"Null Worker Keys: "
    f"{fact_industrial_workforce.filter(col('Worker_Type_Key').isNull()).count()}"
)

print(
    f"Null Industry Keys: "
    f"{fact_industrial_workforce.filter(col('Industry_Key').isNull()).count()}"
)

### 39 — Save Fact Table

In [0]:
(
    fact_industrial_workforce.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "pyspark_dbt.gold.fact_industrial_workforce"
    )
)

print(
    "fact_industrial_workforce created."
)

In [0]:
# diagnostic function
dim_occupation.groupBy(
    "Division",
    "Sub_Division",
    "NCO_Name"
).count().filter(
    col("count") > 1
).show(truncate=False)

## Industrial Occupation Dimension

In [0]:
dim_industrial_occupation = (

    df_ind_main.select(
        "Division",
        "Sub_Division",
        "NCO_Name",
        "Hierarchy_Level"
    )

    .unionByName(

        df_ind_marg.select(
            "Division",
            "Sub_Division",
            "NCO_Name",
            "Hierarchy_Level"
        )

    )

    .distinct()

)

In [0]:
display(dim_industrial_occupation)

In [0]:
window_spec = Window.orderBy(
    "Division",
    "Sub_Division",
    "NCO_Name"
)

dim_industrial_occupation = (

    dim_industrial_occupation

    .withColumn(
        "Industrial_Occupation_Key",
        row_number().over(window_spec)
    )

)

In [0]:
(
    dim_industrial_occupation.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "pyspark_dbt.gold.dim_industrial_occupation"
    )
)